El método de Numerov es un algoritmo numérico de orden 4 especializado en la resolución de ecuaciones diferenciales ordinarias de segundo orden de la forma $y''(x) = f(x)y(x)$. Es particularmente útil en mecánica cuántica para resolver la ecuación de Schrödinger independiente del tiempo, que puede reescribirse en una forma análoga.

La ecuación de Schrödinger unidimensional independiente del tiempo es:
$$-\frac{\hbar^2}{2m}\frac{d^2\psi(x)}{dx^2} + V(x)\psi(x) = E\psi(x)$$
donde $\psi(x)$ es la función de onda, $E$ es la energía total, $V(x)$ es la energía potencial, $m$ es la masa de la partícula y $\hbar$ es la constante de Planck reducida.

Reorganizando la ecuación para que se ajuste a la forma de Numerov, podemos escribir:
$$\frac{d^2\psi(x)}{dx^2} = \frac{2m}{\hbar^2}\left(V(x) - E\right)\psi(x)$$
Definimos $k^2(x) = \frac{2m}{\hbar^2}\left(E - V(x)\right)$, lo que nos permite expresar la ecuación como:
$$\frac{d^2\psi(x)}{dx^2} = -k^2(x)\psi(x)$$
Esta es la forma estándar a la que se aplica el método de Numerov.

El método se basa en una expansión de la función de onda en serie de Taylor. Consideremos los valores de la función de onda en tres puntos discretos y equidistantes: $x_{i-1}$, $x_i$ y $x_{i+1}$, con una separación $\Delta x$. La expansión de Taylor para $\psi(x_{i+1})$ y $\psi(x_{i-1})$ alrededor de $x_i$ es:
$$ \psi_{i+1} = \psi_i + \psi_i' \Delta x + \frac{\psi_i''}{2!}(\Delta x)^2 + \frac{\psi_i'''}{3!}(\Delta x)^3 + \frac{\psi_i''''}{4!}(\Delta x)^4 + O((\Delta x)^5) $$
$$ \psi_{i-1} = \psi_i - \psi_i' \Delta x + \frac{\psi_i''}{2!}(\Delta x)^2 - \frac{\psi_i'''}{3!}(\Delta x)^3 + \frac{\psi_i''''}{4!}(\Delta x)^4 - O((\Delta x)^5) $$
Sumando estas dos expansiones, se cancelan los términos de orden impar:
$$ \psi_{i+1} + \psi_{i-1} = 2\psi_i + \psi_i''(\Delta x)^2 + \frac{\psi_i''''}{12}(\Delta x)^4 + O((\Delta x)^6) $$
De la ecuación de Schrödinger, sabemos que $\psi_i'' = -k_i^2\psi_i$. Para el término de orden superior, diferenciamos la ecuación de Schrödinger:
$$ \psi'''' = \frac{d^2}{dx^2}(-k^2\psi) = - (k^2\psi)'' = - ( (k^2)''\psi + 2(k^2)'\psi' + k^2\psi'' ) $$
Para el método de Numerov, se utiliza la aproximación $\psi_i'''' \approx -k_i^2\psi_i'' = k_i^4\psi_i$. Sustituyendo esto en la suma de las series de Taylor:
$$ \psi_{i+1} + \psi_{i-1} \approx 2\psi_i - k_i^2\psi_i(\Delta x)^2 + \frac{k_i^4\psi_i}{12}(\Delta x)^4 $$
Aunque esta es una aproximación, una manipulación más rigurosa usando la ecuación de Schrödinger y la expansión de Taylor para la segunda derivada lleva a la expresión:
$$ \psi_{i+1}\left(1 + \frac{(\Delta x)^2}{12}k_{i+1}^2\right) + \psi_{i-1}\left(1 + \frac{(\Delta x)^2}{12}k_{i-1}^2\right) = 2\psi_i\left(1 - \frac{5(\Delta x)^2}{12}k_i^2\right) $$
Resolviendo para $\psi_{i+1}$, obtenemos la fórmula de recurrencia de Numerov:
$$ \boxed{\psi_{i+1} = \frac{2\psi_i\left(1 - \frac{5}{12}(\Delta x)^2 k_i^2\right) - \psi_{i-1}\left(1 + \frac{1}{12}(\Delta x)^2 k_{i-1}^2\right)}{1 + \frac{1}{12}(\Delta x)^2 k_{i+1}^2}}$$



El código Python reproduce esta fórmula de recurrencia. La función `numerov_solve` toma como entrada la energía $E$, la función de potencial $V$, y los parámetros físicos.

+ Se calcula el coeficiente $k^2$ para cada punto de la grilla $$ k^2 = \frac{2m}{\hbar^2}(E - V(x)) $$
+ Se inicializa la función de onda $\psi$ con las condiciones de contorno. Para un pozo de potencial con pared infinita en un extremo, se tiene $\psi(0)=0$. Un pequeño valor inicial para $\psi(1)$ permite que el algoritmo inicie la integración.
+ Se aplica la fórmula de recurrencia en un bucle \texttt{for} para calcular $\psi_{i+1}$ a partir de $\psi_i$ y $\psi_{i-1}$ para cada punto $i$.

El valor de la energía $E$ es un parámetro crucial que debe ser ajustado hasta que la función de onda satisfaga la segunda condición de contorno (e.g., $\psi(L)=0$ para el pozo de potencial), lo cual corresponde a un estado ligado. Este proceso generalmente requiere un algoritmo de búsqueda de raíces.


In [13]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import simpson
import ipywidgets as widgets
from IPython.display import display, clear_output

def numerov_solve(E, V_func, x_grid, m=1.0, hbar=1.0):
    """
    Resuelve la ecuación de Schrödinger usando el método de Numerov.
    """
    N = len(x_grid)
    dx = x_grid[1] - x_grid[0]
    
    # Coeficiente k^2 = 2m(E-V)/hbar^2
    k2 = 2 * m * (E - V_func(x_grid)) / (hbar**2)
    
    # Inicializar la función de onda
    psi = np.zeros(N)
    
    # Condiciones iniciales
    psi[0] = 0.0
    psi[1] = 1e-6  # Pequeño valor para iniciar la integración
    
    # Factor de Numerov
    factor = dx**2 / 12.0
    
    # Integración hacia adelante usando el método de Numerov
    for i in range(1, N-1):
        psi[i+1] = (2 * (1 - 5 * factor * k2[i]) * psi[i] - 
                    (1 + factor * k2[i-1]) * psi[i-1]) / (1 + factor * k2[i+1])
    
    return psi

La resolución de la ecuación de Schrödinger no solo implica encontrar la función de onda $\psi(x)$, sino también determinar los valores discretos de energía $E$ para los cuales la solución es físicamente aceptable. Estos valores de $E$ son los **autovalores** del sistema, y las funciones de onda correspondientes $\psi(x)$ son las **autofunciones**. Para un sistema cuántico ligado, como una partícula en un pozo de potencial, las autofunciones deben satisfacer condiciones de contorno específicas, típicamente $\psi(x) \to 0$ en los límites del sistema.

Una propiedad fundamental de las soluciones a la ecuación de Schrödinger es que el número de nodos (puntos donde la función de onda se anula) está directamente relacionado con el nivel de energía del estado. El **teorema de las oscilaciones de Sturm-Liouville** nos dice que la $n$-ésima autofunción (donde $n=1$ es el estado fundamental) tiene exactamente $n-1$ nodos.

+ El estado fundamental ($n=1$) no tiene nodos.
+ El primer estado excitado ($n=2$) tiene un nodo.
+ El $n$-ésimo estado excitado tiene $n-1$ nodos.

Además, los niveles de energía están ordenados de manera creciente: $E_1 < E_2 < E_3 < \dots$. Esto significa que, si aumentamos la energía $E$ en la integración de Numerov, el número de nodos en la función de onda resultante también aumentará. Esta relación monotónica es la base de nuestro algoritmo de búsqueda.

El código utiliza el **método de la bisección**, un algoritmo de búsqueda de raíces robusto y confiable, para encontrar los valores de energía correctos. La "raíz" que buscamos es el valor de $E$ que produce una función de onda con el número de nodos deseado, lo cual es equivalente a la condición de contorno de que la función de onda se anule en el otro extremo del potencial.

El proceso es el siguiente:

+ **Definir un Intervalo de Búsqueda:** Se parte de un intervalo inicial $[E_{\text{low}}, E_{\text{high}}]$ que se cree que contiene el nivel de energía $E_n$ deseado.
+ **Evaluación en el Punto Medio:** Se calcula la energía de prueba $E_{\text{test}} = (E_{\text{low}} + E_{\text{high}}) / 2$.
+ **Resolución y Conteo de Nodos:** Se resuelve la ecuación de Schrödinger con el método de Numerov para $E_{\text{test}}$ y se cuenta el número de nodos (`nodes_test`) de la función de onda resultante.
+ **Ajuste del Intervalo:**

    + Si `nodes_test < n - 1`, significa que la energía $E_{\text{test}}$ es demasiado baja para el estado `n`. Por lo tanto, el nivel de energía correcto debe ser mayor, y se actualiza el límite inferior del intervalo: $E_{\text{low}} = E_{\text{test}}$.
    + Si `nodes_test \ge n - 1`, la energía $E_{\text{test}}$ es igual o mayor que la del estado `n`. Se actualiza el límite superior: $E_{\text{high}} = E_{\text{test}}$.
+ **Iteración:** El proceso se repite, reduciendo el intervalo de búsqueda a la mitad en cada iteración, hasta que la diferencia entre $E_{\text{high}}$ y $E_{\text{low}}$ es menor que una tolerancia predefinida (`tol`).

Este método garantiza la convergencia al valor de energía correcto, siempre y cuando el intervalo inicial contenga el autovalor. El código realiza un primer barrido para encontrar un intervalo inicial adecuado y luego refina la solución con el método de bisección, un enfoque híbrido que aumenta la robustez del algoritmo.

Una vez que se ha encontrado el valor de energía $E_n$ que produce una solución físicamente aceptable, es necesario **normalizar** la función de onda $\psi_n(x)$. La condición de normalización asegura que la probabilidad total de encontrar la partícula en algún lugar del espacio es igual a uno. Matemáticamente, esto se expresa como:
$$ \int_{-\infty}^{\infty} |\psi(x)|^2 dx = 1 $$
En la implementación numérica, la integral se aproxima mediante la **regla de Simpson** o un método similar sobre la grilla discreta de puntos. El código calcula la norma de la función de onda como la raíz cuadrada de esta integral y luego divide la función de onda por esta norma para obtener la versión normalizada:
$$ \psi_{\text{normalizada}}(x) = \frac{\psi(x)}{\sqrt{\int |\psi(x')|^2 dx'}} $$
Esto permite que la función de onda tenga una interpretación probabilística correcta.


In [14]:
def count_nodes(psi):
    """Cuenta el número de nodos (cruces por cero) de la función de onda."""
    return np.sum((psi[:-1] * psi[1:]) < 0)

def find_energy_level(n, V_func, x_grid, m=1.0, hbar=1.0, E_min=0.0, E_max=100.0, tol=1e-6, max_iter=100):
    """
    Encuentra el n-ésimo nivel de energía usando el método de bisección.
    """
    E_low = E_min
    E_high = E_max
    
    # Primero encontramos un rango que contenga el nivel deseado
    for _ in range(max_iter):
        E_test = (E_low + E_high) / 2
        psi_test = numerov_solve(E_test, V_func, x_grid, m, hbar)
        nodes_test = count_nodes(psi_test)
        
        if nodes_test < n - 1:
            E_low = E_test
        else:
            E_high = E_test
        
        if E_high - E_low < tol:
            break
    
    # Refinamos la búsqueda
    E_final = (E_low + E_high) / 2
    psi_final = numerov_solve(E_final, V_func, x_grid, m, hbar)
    
    # Normalizar la función de onda
    norm = np.sqrt(simpson(psi_final**2, x_grid))
    if norm > 0:
        psi_final /= norm
    
    return E_final, psi_final

El código implementa cuatro tipos de potenciales que representan sistemas físicos fundamentales:

+ **Pozo de Potencial Infinito ($V_{\text{pozo}}$):** Es un modelo idealizado donde una partícula está confinada en una región del espacio (entre $-a/2$ y $a/2$). Dentro de esta región, la energía potencial es cero:

\begin{equation}
V(x) = 0, \quad \text{si } |x| \leq \frac{a}{2},
\end{equation}

y fuera de ella es infinita. En el código, se utiliza un valor muy grande ($10^{6}$) en lugar de infinito para la implementación numérica. Este potencial tiene soluciones analíticas conocidas, lo que lo hace ideal para validar el método numérico. La función de onda debe ser cero en las paredes del pozo, lo que solo ocurre para energías discretas.

+ **Potencial Lineal ($V_{\text{lineal}}$):** Este potencial se define como

\begin{equation}
V(x) = \alpha |x|,
\end{equation}

y es una aproximación de una fuerza constante que dirige a la partícula hacia el origen. Este modelo se utiliza, por ejemplo, para describir un quark en un mesón. Las soluciones a este potencial se conocen como las funciones de Airy.

+ **Potencial Cuadrático u Oscilador Armónico ($V_{\text{cuadrático}}$):** Este es uno de los sistemas más importantes en la física, ya que describe muchas interacciones de la naturaleza (vibraciones moleculares, fonones en sólidos, etc.). La energía potencial tiene forma parabólica:

\begin{equation}
V(x) = \frac{1}{2} k x^{2},
\end{equation}

que se asemeja a un resorte cuántico. Sus niveles de energía están espaciados uniformemente.

+ **Potencial de Valor Absoluto ($V_{\text{valor absoluto}}$):** Es matemáticamente similar al potencial lineal, definido como

\begin{equation}
V(x) = \alpha |x|.
\end{equation}

Físicamente, representa una fuerza de restauración que aumenta linealmente con la distancia al origen. Las soluciones a este problema son las mismas que para el potencial lineal.

**Funciones `lambda` y `create_potential_function`:**  
La función `create_potential_function` es un constructor de funciones.  
Toma el tipo de potencial deseado y los parámetros correspondientes, y devuelve una nueva función (creada con `lambda`) que calcula el potencial para cualquier valor de $x$.  

Esto permite que el código sea modular y flexible. En lugar de tener que reescribir la lógica para cada potencial, el programa simplemente puede llamar a `create_potential_function` para obtener la función de potencial correcta y luego pasarla a `numerov_solve`, lo que hace que la interfaz de usuario sea más limpia y fácil de manejar.



In [27]:
# Funciones de potencial
def V_pozo(x, V0=0.0, a=1.0, Vbarrera=1e3):
    """
    Pozo de potencial infinito aproximado.
    Dentro del pozo: V(x) = V0
    Fuera del pozo: V(x) = Vbarrera (grande pero finito)
    """
    return np.where((x >= -a/2) & (x <= a/2), V0, Vbarrera)

def V_lineal(x, alpha=1.0, x0=0.0, V0=0.0):
    """
    Potencial lineal en forma punto-pendiente: V(x) = alpha * (x - x0) + V0
    """
    return alpha * (x - x0) + V0

def V_cuadratico(x, k=1.0, x0=0.0):
    """Potencial cuadrático (oscilador armónico): V(x) = 0.5 * k * (x - x0)^2"""
    return 0.5 * k * (x - x0)**2

def V_valor_absoluto(x, alpha=1.0, x0=0.0):
    """Potencial de valor absoluto: V(x) = alpha * |x - x0|"""
    return alpha * np.abs(x - x0)

def create_potential_function(potential_type, **params):
    """Crea la función de potencial según el tipo seleccionado."""
    if potential_type == "Pozo Infinito":
        return lambda x: V_pozo(x, **params)
    elif potential_type == "Lineal":
        return lambda x: V_lineal(x, **params)
    elif potential_type == "Cuadrático":
        return lambda x: V_cuadratico(x, **params)
    elif potential_type == "Valor Absoluto":
        return lambda x: V_valor_absoluto(x, **params)
    else:
        raise ValueError("Tipo de potencial no reconocido")

def get_energy_guess_range(potential_type, a_pozo=1.0, alpha_lineal=1.0, k_cuadratico=1.0):
    """
    Devuelve un rango de energía apropiado según el tipo de potencial.
    """
    if potential_type == "Pozo Infinito":
        # Energías analíticas para pozo infinito: E_n = (n^2 * π^2 * ħ^2) / (2 * m * a^2)
        # Para n=10 con a=5.0: E ≈ (100 * 9.87) / (2 * 25) ≈ 19.74
        return 0.0, 50.0
    
    elif potential_type == "Lineal":
        # Para potencial lineal, el rango depende de la pendiente
        return 0.0, max(30.0, 10.0 * alpha_lineal)
    
    elif potential_type == "Cuadrático":
        # Oscilador armónico: E_n = ħω(n + 1/2), ω = sqrt(k/m)
        return 0.0, max(25.0, 8.0 * np.sqrt(k_cuadratico))
    
    elif potential_type == "Valor Absoluto":
        return 0.0, max(30.0, 10.0 * alpha_lineal)
    
    return 0.0, 50.0


In [28]:
def update_plot(potential_type, n_level, m, hbar, N, L, 
                a_pozo, V0_pozo, alpha_lineal, x0_lineal, 
                k_cuadratico, x0_cuadratico, alpha_absoluto, x0_absoluto):
    """Actualiza la gráfica con los parámetros seleccionados."""
    
    # Limpiar la salida anterior
    clear_output(wait=True)
    
    # Crear la malla espacial - AJUSTAR SEGÚN EL POTENCIAL
    if potential_type == "Pozo Infinito":
        # Para pozo infinito, usar una malla que se ajuste al ancho del pozo
        x_grid = np.linspace(-a_pozo * 0.8, a_pozo * 0.8, N)
    else:
        # Para otros potenciales, usar la longitud L completa
        x_grid = np.linspace(-L/2, L/2, N)
    
    # Seleccionar parámetros según el tipo de potencial
    if potential_type == "Pozo Infinito":
        params = {'a': a_pozo, 'V0': V0_pozo, 'Vbarrera': 1e3}
    elif potential_type == "Lineal":
        params = {'alpha': alpha_lineal, 'x0': x0_lineal, 'V0': 0.0}
    elif potential_type == "Cuadrático":
        params = {'k': k_cuadratico, 'x0': x0_cuadratico}
    elif potential_type == "Valor Absoluto":
        params = {'alpha': alpha_absoluto, 'x0': x0_absoluto}
    
    # Crear función de potencial
    V_func = create_potential_function(potential_type, **params)
    
    # Calcular el potencial en la malla
    V_values = V_func(x_grid)
    
    # Obtener rango de energía apropiado
    E_min, E_max = get_energy_guess_range(potential_type, a_pozo, alpha_lineal, k_cuadratico)
    
    # Ajustes específicos para el pozo infinito
    if potential_type == "Pozo Infinito":
        # Calcular energía analítica aproximada para guiar la búsqueda
        E_analytical = (n_level**2 * np.pi**2 * hbar**2) / (2 * m * a_pozo**2)
        E_min = max(0.0, E_analytical * 0.8)
        E_max = E_analytical * 1.5
    
    try:
        # Encontrar el nivel de energía
        E_n, psi_n = find_energy_level(n_level, V_func, x_grid, m, hbar, 
                                      E_min=E_min, E_max=E_max)
        
        # Crear la figura
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # ESCALADO ESPECÍFICO PARA CADA POTENCIAL
        if potential_type == "Pozo Infinito":
            # Para pozo infinito: escalar la función de onda para que sea visible
            psi_max = np.max(np.abs(psi_n))
            if psi_max > 0:
                # Escalar la función de onda para que ocupe aproximadamente 1/4 del rango vertical
                V_range = np.max(V_values) - np.min(V_values)
                wave_scale = 0.25 * V_range / psi_max
            else:
                wave_scale = 1.0
            
            # Graficar solo la región del pozo para mejor visualización
            mask = (x_grid >= -a_pozo/2) & (x_grid <= a_pozo/2)
            ax.plot(x_grid[mask], E_n + psi_n[mask] * wave_scale, 'b-', linewidth=2, label=f'ψ_{n_level}(x)')
            ax.plot(x_grid, V_values, 'r-', linewidth=2, label='V(x)')
            
            # Añadir líneas verticales para los bordes del pozo
            ax.axvline(x=-a_pozo/2, color='r', linestyle='--', alpha=0.7)
            ax.axvline(x=a_pozo/2, color='r', linestyle='--', alpha=0.7)
            
        else:
            # Para otros potenciales: escalado normal
            psi_max = np.max(np.abs(psi_n))
            if psi_max > 0:
                wave_scale = 0.3 * (np.max(V_values) - np.min(V_values)) / psi_max
            else:
                wave_scale = 1.0
            
            ax.plot(x_grid, E_n + psi_n * wave_scale, 'b-', linewidth=2, label=f'ψ_{n_level}(x)')
            ax.plot(x_grid, V_values, 'r-', linewidth=2, label='V(x)')
        
        # Graficar el nivel de energía
        ax.axhline(y=E_n, color='g', linestyle='--', linewidth=2, 
                  label=f'E_{n_level} = {E_n:.4f}')
        
        # Configurar la gráfica
        ax.set_xlabel('x', fontsize=14)
        ax.set_ylabel('Energía', fontsize=14)
        
        # Título informativo
        title = f'{potential_type}: n = {n_level}, E = {E_n:.4f}'
        if potential_type == "Pozo Infinito":
            E_analytical = (n_level**2 * np.pi**2 * hbar**2) / (2 * m * a_pozo**2)
            title += f'\nAnalítico: {E_analytical:.4f}, Error: {abs(E_n - E_analytical)/E_analytical*100:.2f}%'
        
        ax.set_title(title, fontsize=16)
        ax.legend(fontsize=12)
        ax.grid(True, alpha=0.3)
        
        # AJUSTAR LÍMITES DE LOS EJES PARA MEJOR VISUALIZACIÓN
        if potential_type == "Pozo Infinito":
            # Para pozo infinito: enfocar en la región del pozo
            ax.set_xlim(-a_pozo * 0.6, a_pozo * 0.6)
            y_max = max(E_n * 1.8, V0_pozo * 1.5)
            ax.set_ylim(-0.1 * y_max, y_max)
        else:
            # Para otros potenciales: ajustar automáticamente
            y_max = max(np.max(V_values), E_n * 1.5)
            y_min = min(np.min(V_values), -0.1 * y_max)
            ax.set_ylim(y_min, y_max * 1.1)
        
        plt.tight_layout()
        plt.show()
        
        print(f"Energía del nivel {n_level}: E = {E_n:.6f}")
        print(f"Número de nodos: {count_nodes(psi_n)}")
        print(f"Rango de búsqueda: [{E_min:.2f}, {E_max:.2f}]")
        
        if potential_type == "Pozo Infinito":
            E_analytical = (n_level**2 * np.pi**2 * hbar**2) / (2 * m * a_pozo**2)
            error_percent = abs(E_n - E_analytical) / E_analytical * 100
            print(f"Energía analítica: {E_analytical:.6f}")
            print(f"Error: {error_percent:.2f}%")
            
    except Exception as e:
        print(f"Error al calcular el nivel de energía: {e}")
        print(f"Rango intentado: [{E_min:.2f}, {E_max:.2f}]")
        print("Sugerencias:")
        print("- Aumentar el número de puntos (N)")
        print("- Ajustar la longitud (L) para el potencial")
        print("- Verificar que el rango de energía sea apropiado")

In [29]:
def create_interactive_interface():
    """Crea la interfaz interactiva completa con ipywidgets."""
    
    # Widgets principales
    potential_dropdown = widgets.Dropdown(
        options=['Pozo Infinito', 'Lineal', 'Cuadrático', 'Valor Absoluto'],
        value='Pozo Infinito',
        description='Potencial:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )
    
    n_slider = widgets.IntSlider(
        value=1,
        min=1,
        max=10,
        step=1,
        description='Nivel n:',
        continuous_update=False,
        style={'description_width': 'initial'}
    )
    
    m_slider = widgets.FloatSlider(
        value=1.0,
        min=0.1,
        max=5.0,
        step=0.1,
        description='Masa (m):',
        continuous_update=False,
        style={'description_width': 'initial'}
    )
    
    hbar_slider = widgets.FloatSlider(
        value=1.0,
        min=0.1,
        max=2.0,
        step=0.1,
        description='ħ:',
        continuous_update=False,
        style={'description_width': 'initial'}
    )
    
    N_slider = widgets.IntSlider(
        value=1000,
        min=500,
        max=5000,
        step=100,
        description='Puntos (N):',
        continuous_update=False,
        style={'description_width': 'initial'}
    )
    
    L_slider = widgets.FloatSlider(
        value=10.0,
        min=2.0,
        max=20.0,
        step=0.5,
        description='Longitud (L):',
        continuous_update=False,
        style={'description_width': 'initial'}
    )
    
    # Widgets específicos para cada potencial
    a_pozo_slider = widgets.FloatSlider(
        value=5.0, min=1.0, max=10.0, step=0.5,
        description='Ancho pozo:', continuous_update=False,
        style={'description_width': 'initial'}
    )
    
    V0_pozo_slider = widgets.FloatSlider(
        value=0.0, min=0.0, max=10.0, step=0.5,
        description='V0 pozo:', continuous_update=False,
        style={'description_width': 'initial'}
    )
    
    alpha_lineal_slider = widgets.FloatSlider(
        value=1.0, min=0.1, max=5.0, step=0.1,
        description='α lineal:', continuous_update=False,
        style={'description_width': 'initial'}
    )
    
    x0_lineal_slider = widgets.FloatSlider(
        value=0.0, min=-2.0, max=2.0, step=0.1,
        description='x0 lineal:', continuous_update=False,
        style={'description_width': 'initial'}
    )
    
    k_cuadratico_slider = widgets.FloatSlider(
        value=1.0, min=0.1, max=5.0, step=0.1,
        description='k cuadrático:', continuous_update=False,
        style={'description_width': 'initial'}
    )
    
    x0_cuadratico_slider = widgets.FloatSlider(
        value=0.0, min=-2.0, max=2.0, step=0.1,
        description='x0 cuadrático:', continuous_update=False,
        style={'description_width': 'initial'}
    )
    
    alpha_absoluto_slider = widgets.FloatSlider(
        value=1.0, min=0.1, max=5.0, step=0.1,
        description='α absoluto:', continuous_update=False,
        style={'description_width': 'initial'}
    )
    
    x0_absoluto_slider = widgets.FloatSlider(
        value=0.0, min=-2.0, max=2.0, step=0.1,
        description='x0 absoluto:', continuous_update=False,
        style={'description_width': 'initial'}
    )
    
    # Contenedores para organizar los widgets
    general_params = widgets.VBox([
        potential_dropdown,
        n_slider,
        m_slider,
        hbar_slider,
        N_slider,
        L_slider
    ], layout=widgets.Layout(width='350px'))
    
    pozo_params = widgets.VBox([
        a_pozo_slider,
        V0_pozo_slider
    ], layout=widgets.Layout(width='350px'))
    
    lineal_params = widgets.VBox([
        alpha_lineal_slider,
        x0_lineal_slider
    ], layout=widgets.Layout(width='350px'))
    
    cuadratico_params = widgets.VBox([
        k_cuadratico_slider,
        x0_cuadratico_slider
    ], layout=widgets.Layout(width='350px'))
    
    absoluto_params = widgets.VBox([
        alpha_absoluto_slider,
        x0_absoluto_slider
    ], layout=widgets.Layout(width='350px'))
    
    # Función para mostrar/ocultar parámetros según el potencial
    def update_parameter_display(change):
        potential_type = change['new']
        pozo_params.layout.display = 'none'
        lineal_params.layout.display = 'none'
        cuadratico_params.layout.display = 'none'
        absoluto_params.layout.display = 'none'
        
        if potential_type == "Pozo Infinito":
            pozo_params.layout.display = 'block'
        elif potential_type == "Lineal":
            lineal_params.layout.display = 'block'
        elif potential_type == "Cuadrático":
            cuadratico_params.layout.display = 'block'
        elif potential_type == "Valor Absoluto":
            absoluto_params.layout.display = 'block'
    
    potential_dropdown.observe(update_parameter_display, names='value')
    update_parameter_display({'new': potential_dropdown.value})
    
    # Área de visualización
    output = widgets.Output()
    
    # Botón de actualización
    update_button = widgets.Button(
        description="Actualizar Gráfica",
        button_style='success',
        layout=widgets.Layout(width='200px')
    )
    
    def on_update_button_clicked(b):
        with output:
            update_plot(
                potential_dropdown.value, n_slider.value, m_slider.value, 
                hbar_slider.value, N_slider.value, L_slider.value,
                a_pozo_slider.value, V0_pozo_slider.value,
                alpha_lineal_slider.value, x0_lineal_slider.value,
                k_cuadratico_slider.value, x0_cuadratico_slider.value,
                alpha_absoluto_slider.value, x0_absoluto_slider.value
            )
    
    update_button.on_click(on_update_button_clicked)
    
    # Diseño final de la interfaz
    params_column = widgets.VBox([
        general_params,
        pozo_params,
        lineal_params,
        cuadratico_params,
        absoluto_params,
        update_button
    ])
    
    interface = widgets.HBox([
        params_column,
        output
    ])
    
    display(interface)
    
    # Actualizar inicialmente
    with output:
        update_plot(
            potential_dropdown.value, n_slider.value, m_slider.value, 
            hbar_slider.value, N_slider.value, L_slider.value,
            a_pozo_slider.value, V0_pozo_slider.value,
            alpha_lineal_slider.value, x0_lineal_slider.value,
            k_cuadratico_slider.value, x0_cuadratico_slider.value,
            alpha_absoluto_slider.value, x0_absoluto_slider.value
        )

# Ejecutar la interfaz interactiva
if __name__ == "__main__":
    create_interactive_interface()